In [1]:
! pip3 install tensorflow tensorflow-hub numpy 
import os
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os
import re
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType, connections, utility


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Download the model and load the model

module_url = "https://tfhub.dev/google/universal-sentence-encoder-large/5" #@param ["https://tfhub.dev/google/universal-sentence-encoder/4", "https://tfhub.dev/google/universal-sentence-encoder-large/5"]
model = hub.load(module_url)

In [3]:
# Function to generate embeddings

def embeddings(text):
    return np.array(model(text)).flatten().tolist()

In [5]:
# Dataset available at https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection
# Download the dataset and extract the files before running this cell
# We will create the embeddings from this dataset using the universal-sentence-encoder model

file_path = 'SMSSpamCollection'

# Fix encoding issue - SMSSpamCollection uses tab-separated format and may have encoding issues
with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
    lines = [line for line in file]

# Parse tab-separated values (format: ham/spam\tmessage)
msgs = [x.split('\t')[1].replace('\n', '') if '\t' in x else x.split(',')[1].replace('\n', '') for x in lines]
embdngs = [embeddings([x]) for x in msgs]
indx = list(range(1, len(msgs)+1))

data_to_insert = [indx, msgs, embdngs]

In [6]:
# Connect to milvus database

connections.connect(
  alias="default",
  host='localhost',
  port='19530'
)

In [7]:
# Field Schema
id = FieldSchema(
  name="id",
  dtype=DataType.INT64,
  is_primary=True,
)
message = FieldSchema(
  name="message",
  dtype=DataType.VARCHAR,
  max_length=6000,
)
message_vec = FieldSchema(
  name="message_embeddings",
  dtype=DataType.FLOAT_VECTOR,
  dim=512
)
# collection schema
collection_schema = CollectionSchema(
  fields=[id, message, message_vec],
  description="Spam SMS collection"
)
# Create collection
collection = Collection(
    name="Spam_Test",
    schema=collection_schema,
    using='default')
utility.list_collections()

['dynamic_schema_example', 'partition_key_collection', 'Spam_Test', 'Album1']

In [8]:
# Insert entities

data_insert = collection.insert(data_to_insert)

In [9]:
# Create Index
index_params = {
  "metric_type":"L2",
  "index_type":"IVF_FLAT",
  "params":{"nlist":1024},
  "index_name": "SMS_IVF_FLAT_TEST"
}

# Index on vector field
collection.create_index(
  field_name="message_embeddings", 
  index_params=index_params
)

Status(code=0, message=)

In [10]:
# Load the collection

collection.load(replica_number=1)

In [11]:
# test message
test_message = ["claim prize"]
test_message_vector = embeddings(test_message)

In [12]:
## Vector Similarity Search
search_params = {"metric_type": "L2", "params": {"nprobe": 64}}

results = collection.search(
	data=[test_message_vector], 
	anns_field="message_embeddings", 
	param=search_params,
	limit=5, 
	expr=None,
	output_fields=['message']
)

for result in results[0]:
    print (result)

id: 4049, distance: 1.2084664106369019, entity: {'message': 'Win a 1000 cash prize or a prize worth 5000'}
id: 2222, distance: 1.2359904050827026, entity: {'message': 'You have WON a guaranteed 1000 cash or a 2000 prize. To claim yr prize call our customer service representative on 08714712379 between 10am-7pm Cost 10p'}
id: 5014, distance: 1.2392092943191528, entity: {'message': 'You have WON a guaranteed 1000 cash or a 2000 prize. To claim yr prize call our customer service representative on 08714712412 between 10am-7pm Cost 10p'}
id: 1876, distance: 1.2614030838012695, entity: {'message': 'You have WON a guaranteed 1000 cash or a 2000 prize.To claim yr prize call our customer service representative on'}
id: 10, distance: 1.2638798952102661, entity: {'message': 'WINNER!! As a valued network customer you have been selected to receivea 900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.'}
